In [25]:
from sage.symbolic.integration.integral import definite_integral
import sympy


x, y, z = var("x"), var("y"), var("z")
definite_integral(1,y,x,z)


def integral_over_tet(expr, x,y,z, h):
    a = definite_integral(expr, y, x, z)
    b = definite_integral(a, z, x, h)
    c = definite_integral(b, x, 0, h)
    return c

def integral_unit_cube_square(expr, x, y ,z, h):
    a = definite_integral(expr*expr, x,0,h)
    b= definite_integral(a, y,0,h)
    c= definite_integral(b, z,0,h)

    return c

def linear_expression(coeffs, x, y, z, h):
    a = coeffs[0]*(1-z/h)
    b = coeffs[1]*(z/h-y/h)
    c = coeffs[2]*(y/h-x/h)
    d = coeffs[3]*x/h
    return a+b+c+d

def integral_over_tet_substract(corner_list, expr, x, y, z):
    linear_function = corner_list[0]*(1-z-y)+corner_list[1]*(z-y) + corner_list[2]*(y-x) + corner_list[3]*x
    return integral_over_tet(expr*linear_function, x, y, z)
    


def integral_over_cube(expr, x, y, z):  
    return sum(integral_over_tet(expr, *mut) for mut in perm.list())


last_bit = lambda b: int(log(b, 2))+1
first_bit = lambda b: (b&-b).bit_length()

def integral_over_linears(coeff_list, x, y, z):
    perm = Permutations([x, y, z])
    variables = perm.list()
    #print("Num variables ", len(variables))
    counter = 0
    result = 0
    for i in [3,5,6]:
        for k in [2**(first_bit(i)-1), 2**(last_bit(i)-1)]:
            coeffs = [coeff_list[0], coeff_list[k], coeff_list[i], coeff_list[7]]
            expr = linear_expression(coeffs, *(variables[counter]))
            a = integral_over_tet(expr, *(variables[counter]))
            result += a
            counter+=1

    return result

def deviation_integral(coeff_list, expr, x, y, z, h):
    perm = Permutations([x, y, z])
    variables = perm.list()
    #print("Num variables ", len(variables))
    counter = 0
    result = 0
    for i in [3,5,6]:
        for k in [2**(first_bit(i)-1), 2**(last_bit(i)-1)]:
            coeffs = [coeff_list[0], coeff_list[k], coeff_list[i], coeff_list[7]]
            expr_new = expr-linear_expression(coeffs, *(variables[counter]), h)
            #print(expr_new)
            a = integral_over_tet(expr_new*expr_new, *(variables[counter]), h)
            result += a
            counter+=1
    #print(expr_new)

    return result

def compute_gradient(expr):
    return vector(
        [derivative(expr, x),
         derivative(expr, y),
         derivative(expr, z)]
    )

def energy_norm_integral(coeff_list, expr, x, y, z, h):
    perm = Permutations([x, y, z])
    variables = perm.list()
    #print("Num variables ", len(variables))
    counter = 0
    result = 0
    for i in [3,5,6]:
        for k in [2**(first_bit(i)-1), 2**(last_bit(i)-1)]:
            coeffs = [coeff_list[0], coeff_list[k], coeff_list[i], coeff_list[7]]
            expr_new = expr-linear_expression(coeffs, *(variables[counter]), h)
            expr_new_grad = compute_gradient(expr_new)
            a = integral_over_tet(expr_new_grad*expr_new_grad, *(variables[counter]), h)
            result += a
            counter+=1
    #print(expr_new)

    return result
    

In [18]:
result = integral_unit_cube_square(2*x**3+2*y**2+3*z-x, x, y, z, 1)
float(result)

5.904761904761905

In [27]:
a, b, c, d, e, f, g, i = var("a"), var("b"), var("c"), var("d"), var("e"), var("f"), var("g"), var("i")
l, m, n, h =var("l,m,n, h")
dev_int=deviation_integral([a, b, c, d, e, f, g, i] , -3*(x + l)**2-4*(y+m)**2+7*(z+n)**2, x,y,z, h).full_simplify()
dev_func= dev_int.function(a,b,c,d,e,f,g,i,l, m, n, h)
#print(sympy.ccode(dev_int))

energy_norm_int = energy_norm_integral([a, b, c, d, e, f, g, i] , -3*(x + l)**2-4*(y+m)**2+7*(z+n)**2, x,y,z, h).full_simplify()
print(sympy.ccode(energy_norm_int))

(296.0/3.0)*pow(h, 5) + 36*pow(h, 3)*pow(l, 2) + 64*pow(h, 3)*pow(m, 2) + 196*pow(h, 3)*pow(n, 2) - 2.0/3.0*pow(h, 3)*(14*b - 8*c + 3*d - 6*e + 4*f - 7*g) + h*pow(i, 2) - 2.0/3.0*h*i*(d + f + g) + (1.0/3.0)*h*(3*pow(a, 2) - 2*a*b - 2*a*c - 2*a*e + 2*pow(b, 2) + 2*pow(c, 2) + 2*pow(d, 2) - d*(b + c) + 2*pow(e, 2) + 2*pow(f, 2) - f*(b + e) + 2*pow(g, 2) - g*(c + e)) + 2*l*(18*pow(h, 4) + 2*pow(h, 2)*i - pow(h, 2)*(2*a + b + c + 2*d - 2*e - f - g)) + (8.0/3.0)*m*(24*pow(h, 4) + 2*pow(h, 2)*i - pow(h, 2)*(2*a + b - 2*c - d + e + 2*f - g)) + (14.0/3.0)*n*(42*pow(h, 4) - 2*pow(h, 2)*i + pow(h, 2)*(2*a - 2*b + c - d + e - f + 2*g))


In [23]:
compute_gradient(x+y+z)*compute_gradient(x+y+z)

3

In [ ]:
e